# QuantaEdge AI Tutor — LoRA fine-tuning pilot
This notebook trains an adapter only. It expects reviewed train/validation/test JSONL files committed under `data/ai-training/datasets/`. Do not use draft records for student deployment. Use a GPU runtime and save checkpoints to Google Drive.

In [ ]:
!nvidia-smi || true
!pip -q install "transformers>=4.51.0" "trl>=0.17.0" "peft>=0.15.0" "datasets>=3.0.0" accelerate bitsandbytes huggingface_hub

In [ ]:
from pathlib import Path
import json, hashlib, os
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/QuantaEdge'
if not Path(REPO).exists():
    !git clone https://github.com/Manav326/QuantaEdge.git /content/QuantaEdge
%cd $REPO
DATA = Path('data/ai-training/datasets')
for name in ('train','validation','test'):
    path = DATA / f'{name}.jsonl'
    if not path.exists(): raise FileNotFoundError(f'Missing {path}; generate and review dataset first.')
manifest = json.loads((DATA/'manifest.json').read_text(encoding='utf-8'))
print('dataset records:', manifest.get('records'), 'splits:', manifest.get('splits'))
if manifest.get('status') != 'APPROVED_ONLY':
    raise RuntimeError('Refusing to train: manifest is not APPROVED_ONLY. Review/approve the data first.')
digest = hashlib.sha256((DATA/'dataset_master.jsonl').read_bytes()).hexdigest()
print('dataset sha256:', digest)

In [ ]:
from datasets import load_dataset
files = {k: str(DATA/f'{k}.jsonl') for k in ('train','validation','test')}
dataset = load_dataset('json', data_files=files)
print(dataset)

In [ ]:
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > GPU before training.')
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
MODEL_ID = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map='auto')
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('loaded', MODEL_ID, 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
OUTPUT = '/content/drive/MyDrive/QuantaEdge/ai-adapter-pilot'
training_args = SFTConfig(
    output_dir=OUTPUT,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    max_length=1024,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to='none',
    packing=False,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM'),
)
trainer.train(resume_from_checkpoint=True if any(Path(OUTPUT).glob('checkpoint-*')) else None)
trainer.save_model(OUTPUT)
tokenizer.save_pretrained(OUTPUT)
print('adapter saved to persistent Drive:', OUTPUT)

In [ ]:
# Basic smoke test only. Complete the human-reviewed held-out evaluation before deployment.
prompt = 'Explain one-half and one-quarter using a simple example.'
messages = [{'role':'user','content':prompt}]
rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(rendered, return_tensors='pt').to(model.device)
with torch.no_grad(): output = model.generate(**inputs, max_new_tokens=180, do_sample=False)
print(tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
print('Before release, evaluate Maths correctness, Science accuracy, Hindi/English fluency and held-out chapter generalization; compare against the un-finetuned base model.')

## Release checklist
- Manifest status is `APPROVED_ONLY` and all source reuse rights are documented.
- Dataset split integrity and validation tests pass.
- Compare baseline vs adapter on the held-out test split; manually verify calculations, scientific claims, Hindi/English quality and student-safety behavior.
- Save model revision, dataset SHA-256, notebook version, hyperparameters, metrics and checkpoint location.
- Do not commit checkpoints or credentials to the source repository; store them in access-controlled artifact storage.